# Efecto de capacitación sobre ingresos: estimación ajustada

Consulta la [metodología](../docs/metodologia.md) y ejecuta las etapas en orden. Los comentarios se calculan con los resultados guardados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("training_program_impact")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from training_program_impact.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from training_program_impact.analysis import run_stage
import pandas as pd
metrics,comment=run_stage(PROJECT_ROOT,"modelado")
display(Markdown(comment))
scalars={k:v for k,v in metrics.items() if not isinstance(v,(dict,list))}
if scalars:display(pd.DataFrame(list(scalars.items()),columns=["Indicador","Valor"]))
if "candidatos" in metrics:display(pd.DataFrame(metrics["candidatos"]))
for filename in ["eda.png","evaluacion.png"]:
    path=PROJECT_ROOT/"reports/modelado"/filename
    if path.exists():display(Image(filename=str(path)))
for filename in ["estimaciones.csv","metricas_test.csv","horas_por_empleado.csv"]:
    path=PROJECT_ROOT/"reports/modelado"/filename
    if path.exists():display(pd.read_csv(path).head(20))

# Efecto de capacitación sobre ingresos: modelado

Ajuste preespecificado por covariables previas e interacciones: **1,621.58 USD**, IC 95% HC3 **[212.54, 3,030.62]**. El coeficiente de asignación se interpreta respecto a covariables centradas en esta muestra. No se selecciona el estimador por producir un efecto mayor o un intervalo que excluya cero.


,Indicador,Valor
0,efecto_usd,1621.583101
1,error_estandar_hc3,716.872751
2,ci_inferior,212.544498
3,ci_superior,3030.621704
4,confidence_level,0.95
5,parametros,18
6,metodo,OLS Lin con interacciones y errores HC3


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/04_modelado.json")
from training_program_impact.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))

[Abrir informe integral](../reports/informe_final/informe_final.md)